In [2]:
!pip install pandas numpy scikit-learn

In [3]:
import pandas as pd
import numpy as np
import os

# Đảm bảo thư mục data tồn tại
os.makedirs('../data', exist_ok=True)

def generate_synthetic_data(num_samples=500000):
    print(f"Đang tạo {num_samples} mẫu dữ liệu tổng hợp...")
    np.random.seed(42)
    
    # 1. Các thuộc tính giao thoa (Intersectional attributes)
    gender = np.random.choice(['Male', 'Female'], size=num_samples)
    race = np.random.choice(['White', 'Black', 'Asian', 'Hispanic'], size=num_samples, p=[0.5, 0.2, 0.15, 0.15])
    age = np.random.randint(18, 70, size=num_samples)
    education_years = np.random.normal(12, 3, size=num_samples).clip(8, 20)
    
    # 2. Tạo Bias Pathway (Đường dẫn thiên kiến nhân quả)
    # Thu nhập (Income) bị ảnh hưởng trực tiếp bởi gender và race (đây là bias pathway ta cần thuật toán tự động tìm ra ở Bước 3)
    base_income = 30000 + (education_years * 3000) + (age * 200)
    gender_penalty = np.where(gender == 'Female', -5000, 0)
    race_penalty = np.where(np.isin(race, ['Black', 'Hispanic']), -4000, 0)
    
    income = base_income + gender_penalty + race_penalty + np.random.normal(0, 5000, size=num_samples)
    
    # 3. Biến mục tiêu: Phê duyệt tín dụng (Credit Approval)
    # Xác suất duyệt phụ thuộc vào income. Vì income có bias, kết quả duyệt cũng sẽ có intersectional bias.
    prob_approval = 1 / (1 + np.exp(-(income - 45000) / 10000))
    loan_approved = np.random.binomial(1, prob_approval)
    
    df = pd.DataFrame({
        'gender': gender,
        'race': race,
        'age': age,
        'education_years': education_years.astype(int),
        'income': income.round(2),
        'loan_approved': loan_approved
    })
    
    # Lưu ra file CSV
    file_path = '../data/synthetic_intersectional_data.csv'
    df.to_csv(file_path, index=False)
    print(f"✅ Đã lưu dữ liệu tổng hợp tại: {file_path}")
    
    return df.head()

# Thực thi hàm
df_synthetic = generate_synthetic_data()
df_synthetic

Đang tạo 500000 mẫu dữ liệu tổng hợp...
✅ Đã lưu dữ liệu tổng hợp tại: ../data/synthetic_intersectional_data.csv


,gender,race,age,education_years,income,loan_approved
0,Male,Hispanic,69,10,66197.79,1
1,Female,White,66,8,72311.32,1
2,Male,White,37,12,72955.85,1
3,Male,Black,33,14,69839.77,1
4,Male,White,45,9,69897.16,1


In [4]:
import pandas as pd
import urllib.request
import os

# --- 1. Xử lý German Credit Data ---
print("Đang tải và xử lý German Credit Data...")
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data"
columns = ['status', 'duration', 'credit_history', 'purpose', 'credit_amount', 
           'savings', 'employment', 'installment_rate', 'personal_status_sex', 
           'other_debtors', 'residence_since', 'property', 'age', 'other_installment_plans', 
           'housing', 'existing_credits', 'job', 'num_dependents', 'telephone', 'foreign_worker', 'target']

df_german = pd.read_csv(url, sep=' ', header=None, names=columns)

# Chuyển đổi giới tính từ mã của UCI và target (1: Tốt, 0: Xấu)
df_german['gender'] = df_german['personal_status_sex'].apply(lambda x: 'Female' if x in ['A92', 'A95'] else 'Male')
df_german['target'] = df_german['target'].apply(lambda x: 1 if x == 1 else 0) 
df_german.to_csv('../data/german_credit_processed.csv', index=False)
print("✅ Đã lưu German Credit Data.")

# --- 2. Xử lý Home Credit Data ---
file_path = '../data/application_train.csv'
if os.path.exists(file_path):
    print("Đang xử lý Home Credit Data...")
    df_home = pd.read_csv(file_path)
    
    # Lấy các cột quan trọng cho fairness
    cols_to_keep = ['TARGET', 'CODE_GENDER', 'AMT_INCOME_TOTAL', 'DAYS_BIRTH', 'NAME_EDUCATION_TYPE']
    df_home = df_home[cols_to_keep].copy()
    
    # Đổi DAYS_BIRTH thành tuổi (age)
    df_home['age'] = (df_home['DAYS_BIRTH'] / -365).astype(int)
    df_home.drop(columns=['DAYS_BIRTH'], inplace=True)
    df_home = df_home[df_home['CODE_GENDER'] != 'XNA'] # Xóa giới tính không xác định
    
    df_home.to_csv('../data/home_credit_processed.csv', index=False)
    print("✅ Đã lưu Home Credit Data.")
    display(df_home.head(3))
else:
    print(f"❌ KHÔNG TÌM THẤY FILE! Vui lòng tải application_train.csv từ Kaggle và đặt vào thư mục: {file_path}")

Đang tải và xử lý German Credit Data...
✅ Đã lưu German Credit Data.
Đang xử lý Home Credit Data...
✅ Đã lưu Home Credit Data.


,TARGET,CODE_GENDER,AMT_INCOME_TOTAL,NAME_EDUCATION_TYPE,age
0,1,M,202500.0,Secondary / secondary special,25
1,0,F,270000.0,Higher education,45
2,0,M,67500.0,Secondary / secondary special,52
